<a href="https://colab.research.google.com/github/Alijewa25/Alijewa25/blob/main/4Motion_T%C9%99hmin%C9%99%C6%8Fliyeva.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!pip install geopandas fiona shapely -q

import zipfile
import os
import warnings
import geopandas as gpd
import pandas as pd
import fiona

warnings.filterwarnings('ignore')

fiona.drvsupport.supported_drivers['KML'] = 'rw'
fiona.drvsupport.supported_drivers['LIBKML'] = 'rw'

print("Kitabxanalar hazır.")

Kitabxanalar hazır.


In [10]:
def read_kmz(file_path):
    folder = f"temp_{os.path.basename(file_path).split('.')[0]}"
    with zipfile.ZipFile(file_path, 'r') as zip_ref:
        zip_ref.extractall(folder)

    kml_file = [os.path.join(r, f) for r, _, files in os.walk(folder) for f in files if f.endswith('.kml')][0]
    layers = fiona.listlayers(kml_file)
    gdfs = [gpd.read_file(kml_file, layer=l, driver='KML') for l in layers]
    res = pd.concat([g for g in gdfs if not g.empty], ignore_index=True)
    if res.crs is None:
        res.set_crs(epsg=4326, inplace=True)
    return res

gdf_zolaq = read_kmz("movcud-zolaqlar-velo.kmz")
gdf_parkinq = read_kmz("mpv-v-mps.-son.kmz")

In [11]:
gdf_parkinq_points = gdf_parkinq[gdf_parkinq.geometry.type.isin(['Point', 'Point Z'])].copy()

baku_points = gdf_parkinq_points[
    (gdf_parkinq_points.geometry.y >= 40.0) & (gdf_parkinq_points.geometry.y <= 40.7) &
    (gdf_parkinq_points.geometry.x >= 49.5) & (gdf_parkinq_points.geometry.x <= 50.5)
].copy()

mpv_parkings = baku_points[baku_points['Name'].astype(str).str.contains('MPV', case=False, na=False)].copy()

gdf_zolaq_32639 = gdf_zolaq.to_crs(epsg=32639)
mpv_32639 = mpv_parkings.to_crs(epsg=32639)

print(f"MPV parkinq sayı: {len(mpv_32639)}")

MPV parkinq sayı: 185


## **Hissə 1: Dəqiq Geofəza Hesablamaları**

### Sual 1.1
Mövcud velosiped zolaqlarının (Mövcud zolaqlar qatı, EPSG:32639 proyeksiyası ilə) ümumi uzunluğu neçə kilometrdir (vergüldən sonra 2 rəqəm) və ən çox velosiped zolağı olan küçə/prospekt hansıdır?

*(Format: "Uzunluq km, Küçə adı")*

In [12]:
gdf_zolaq_32639['length_m'] = gdf_zolaq_32639.geometry.length
total_len_km = round(gdf_zolaq_32639['length_m'].sum() / 1000, 2)
street_col = 'Name' if 'Name' in gdf_zolaq_32639.columns else gdf_zolaq_32639.columns[0]
top_street = gdf_zolaq_32639.groupby(street_col)['length_m'].sum().idxmax()

print(f"{total_len_km}, {top_street}")

57.14, 8 Noyabr pr.


### Sual 1.2
Velosiped məntəqələri yeni qatındakı 185 velosiped parkinqindən neçəsi mövcud velosiped zolaqlarına 50 metr və daha yaxın məsafədə yerləşir?

*(Format: Yalnız tam ədəd daxil edin)*

In [13]:
zolaq_buffer_50m = gdf_zolaq_32639.geometry.buffer(50).union_all()
within_50m_count = int(mpv_32639.geometry.within(zolaq_buffer_50m).sum())

print(within_50m_count)

87


### Sual 1.3
Parkinq koordinatlarını araşdırdıqda Bakı mərkəzi şəhər çərçivəsindən (enlik 40.30–40.46, uzunluq 49.70–50.05) kənarda qalan neçə parkinq məntəqəsi var və adlarını qeyd edin?

*(Format: "Say: Məntəqə adları")*

In [14]:
mpv_4326 = mpv_parkings.to_crs(epsg=4326)
mpv_4326['lon'] = mpv_4326.geometry.x
mpv_4326['lat'] = mpv_4326.geometry.y

outside_mpv = mpv_4326[
    (mpv_4326['lat'] < 40.30) | (mpv_4326['lat'] > 40.46) |
    (mpv_4326['lon'] < 49.70) | (mpv_4326['lon'] > 50.05)
]

count_outside = len(outside_mpv)
names_outside = ", ".join(outside_mpv['Name'].astype(str).tolist())

print(f"{count_outside}: {names_outside}")

4: MPV-144-2, MPV-145-2, MPV-146-2, MPV-147-2


## **Hissə 2: Strateji Təhlil və AYNA Rəhbərliyinə Təklif**

**Sual:** AYNA rəhbərliyinə bu "tənha" parkinqləri şəbəkəyə ən tez qoşmaq üçün növbəti mərhələdə yeni velosiped zolağının salınmasını hansı əsas istiqamət/küçə üzrə təklif edərdiniz və niyə?



**Cavab:**
Mərkəzdən kənarda qalan bu 4 parkinqi (MPV-144-2 - MPV-147-2) ümumi şəbəkəyə qatmaq üçün növbəti velosiped zolağının Heydər Əliyev prospekti boyunca salınmasını təklif edərdik.

Bunun əsas səbəbi həmin istiqamətin həm əhalinin sıx yaşadığı ərazi olması, həm də mərkəzə ən çox hərəkət olan yollardan biri olmasıdır. Yeni zolaq kənarda qalan, istifadəsiz görünən bu parkinqləri canlandıracaq və insanların təhlükəsiz şəkildə birbaşa şəhərin mərkəzi velosiped yollarına çıxışını təmin edəcək.